# Disease Prediction from Medical Data

This notebook builds a beginner-friendly machine learning project to predict whether a patient has breast cancer using the Breast Cancer dataset from Scikit-Learn.

## Goals
- Load the dataset
- Perform exploratory data analysis (EDA)
- Train multiple classification models
- Compare model performance
- Save the best model for future prediction

In [ ]:
# Import required libraries
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

# Make plots look cleaner
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

In [ ]:
# Define project folders so the notebook can save outputs in the correct place
current_dir = Path.cwd()
base_dir = current_dir.parent if current_dir.name == "notebooks" else current_dir
data_dir = base_dir / "data"
models_dir = base_dir / "models"
screenshots_dir = base_dir / "screenshots"

for directory in [data_dir, models_dir, screenshots_dir]:
    directory.mkdir(parents=True, exist_ok=True)

base_dir

In [ ]:
# Load the Breast Cancer dataset from Scikit-Learn
dataset = load_breast_cancer()

# Convert features into a pandas DataFrame
X = pd.DataFrame(dataset.data, columns=dataset.feature_names)

# Convert target into a pandas Series
y = pd.Series(dataset.target, name="target")

# Create a complete dataset for easier analysis
df = X.copy()
df["target"] = y

df.head()

## Exploratory Data Analysis

In [ ]:
# Display dataset shape
print("Feature matrix shape:", X.shape)
print("Dataset shape with target column:", df.shape)

# Display feature names
print("\nFeature names:")
for feature_name in dataset.feature_names:
    print("-", feature_name)

In [ ]:
# Display class distribution with counts and percentages
class_names = {0: dataset.target_names[0].title(), 1: dataset.target_names[1].title()}
class_distribution = (
    y.map(class_names)
    .value_counts()
    .rename_axis("class_name")
    .reset_index(name="count")
)
class_distribution["percentage"] = (
    class_distribution["count"] / len(y) * 100
).round(2)

class_distribution

In [ ]:
# Perform missing value analysis
missing_values = df.isnull().sum()
missing_summary = pd.DataFrame(
    {
        "missing_count": missing_values,
        "missing_percentage": (missing_values / len(df) * 100).round(2),
    }
)

missing_summary

In [ ]:
# Visualize class distribution
plt.figure(figsize=(8, 5))
target_labels = y.map(class_names)
sns.countplot(x=target_labels, palette="Set2", hue=target_labels, legend=False)
plt.title("Class Distribution")
plt.xlabel("Class")
plt.ylabel("Number of Samples")
plt.tight_layout()
plt.show()

In [ ]:
# Plot a correlation heatmap for all input features
plt.figure(figsize=(18, 14))
correlation_matrix = X.corr()
sns.heatmap(
    correlation_matrix,
    cmap="coolwarm",
    linewidths=0.3,
    square=True,
    cbar_kws={"shrink": 0.75},
)
plt.title("Correlation Heatmap of Breast Cancer Features")
plt.tight_layout()
plt.show()

## Preprocessing

- There are no missing values in this dataset.
- No encoding is required because all features are already numeric.
- Standard scaling is used only for Logistic Regression.

In [ ]:
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)

In [ ]:
# Create the three required machine learning models
models = {
    "Logistic Regression": Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=1000, solver="liblinear", random_state=42)),
        ]
    ),
    "Decision Tree": DecisionTreeClassifier(random_state=42, max_depth=5),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
}

models

In [ ]:
# Train each model and store evaluation metrics
results = []
trained_models = {}
roc_curve_data = {}

for model_name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]

    trained_models[model_name] = model
    roc_curve_data[model_name] = (roc_curve(y_test, probabilities), probabilities)

    results.append(
        {
            "Model": model_name,
            "Accuracy": accuracy_score(y_test, predictions),
            "Precision": precision_score(y_test, predictions, zero_division=0),
            "Recall": recall_score(y_test, predictions, zero_division=0),
            "F1 Score": f1_score(y_test, predictions, zero_division=0),
            "ROC-AUC": roc_auc_score(y_test, probabilities),
        }
    )

results_df = pd.DataFrame(results).sort_values(by=["ROC-AUC", "Accuracy"], ascending=False)
results_df

In [ ]:
# Visualize model comparison with a bar chart
comparison_plot_df = results_df.set_index("Model")[["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"]]
comparison_plot_df.plot(kind="bar", figsize=(12, 6), ylim=(0.85, 1.02))
plt.title("Model Performance Comparison")
plt.xlabel("Model")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Select the best model based on ROC-AUC
best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]

print("Best model:", best_model_name)
print(results_df.iloc[0])

In [ ]:
# Plot the confusion matrix of the best model
best_predictions = best_model.predict(X_test)
conf_matrix = confusion_matrix(y_test, best_predictions)

plt.figure(figsize=(6, 5))
sns.heatmap(
    conf_matrix,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=[name.title() for name in dataset.target_names],
    yticklabels=[name.title() for name in dataset.target_names],
)
plt.title("Confusion Matrix of Best Model")
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.tight_layout()
plt.show()

In [ ]:
# Plot ROC curves for all models
plt.figure(figsize=(8, 6))

for model_name, (curve_values, probabilities) in roc_curve_data.items():
    false_positive_rate, true_positive_rate, _ = curve_values
    auc_score = roc_auc_score(y_test, probabilities)
    plt.plot(false_positive_rate, true_positive_rate, label=f"{model_name} (AUC = {auc_score:.4f})")

plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random Guess")
plt.title("ROC Curve Comparison")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
# Save the best model so it can be reused later
joblib.dump(best_model, models_dir / "best_model.pkl")

model_metadata = {
    "best_model_name": best_model_name,
    "feature_names": [str(name) for name in dataset.feature_names],
    "target_names": [str(name) for name in dataset.target_names],
    "test_size": 0.20,
    "random_state": 42,
    "selection_metric": "ROC-AUC",
}

joblib.dump(model_metadata, models_dir / "model_metadata.pkl")
results_df.to_csv(data_dir / "model_comparison.csv", index=False)

test_data = X_test.copy()
test_data["target"] = y_test.values
test_data.to_csv(data_dir / "test_data.csv", index=False)

print("Saved model and related files successfully.")

In [ ]:
# Make a sample prediction using the best model
sample_index = 0
sample_input = pd.DataFrame([dataset.data[sample_index]], columns=dataset.feature_names)
sample_prediction = best_model.predict(sample_input)[0]
sample_probabilities = best_model.predict_proba(sample_input)[0]

print("Predicted class:", dataset.target_names[sample_prediction].title())
print("Actual class:", dataset.target_names[dataset.target[sample_index]].title())
print("Probabilities:")

for class_name, probability in zip(dataset.target_names, sample_probabilities):
    print(f"- {class_name.title()}: {probability:.4f}")

## Conclusion

This notebook demonstrated a complete end-to-end machine learning workflow for disease prediction using the Breast Cancer dataset. Among the three models tested, **Logistic Regression** performed the best on the test set and was saved for future use.